In [1]:
from utils import *
from utils.new_custom_classes import ParameterField
from utils.sigmav_functions import *
import numpy as np
import matplotlib.pyplot as plt
import itertools
import plotly.graph_objects as go
from tqdm import tqdm
from scipy.integrate import solve_ivp

import pandas as pd
import seaborn as sns

# Parametrization points

In [2]:
# generic number of values for each parameter used for the parametric analysis
param_points = 3

# OPTIONAL - if not specified, the time necessary to reach a 50%D-50%T concentration in the plasma will be calculated
# total time for the parametric analysis [s]
total_time = 10* u.yr

# Parameters

### Plasma parameters

In [ ]:
V_plasma_field = ParameterField(
    parametrization_type="normal", mean=150, std=15, unit=u.m**3, 
    param_points=1, name="plasma_volume"
)

T_i_field = ParameterField(
    parametrization_type="linear", min_val=14, max_val=20, unit=u.keV,
    param_points=3, name="T_i_field"
)

n_tot_field = ParameterField(
    parametrization_type="linear", min_val=1.3e20, max_val=2.1e20, unit=u.m**(-3),
    param_points=3, name="n_tot_field"
)

tau_p_T_field = ParameterField(
    parametrization_type="linear", min_val = 0.1, max_val=0.1, unit=u.s,
    param_points=1, name="tau_p_T"
)

tau_p_He3_field = ParameterField(
    parametrization_type="normal", mean=1, std=0.5, unit=u.s,
    param_points=1, name="tau_p_He3"
)

P_aux_field = ParameterField(
    parametrization_type="linear", min_val=20, max_val=100, unit=u.MW,
    param_points=1, name="P_aux"
)

P_lost_rad_field = ParameterField(
    parametrization_type="linear", min_val=0, max_val=20, unit=u.MW,
    param_points=param_points, name="P_lost_rad"
)

P_aux_all_DT_field = ParameterField(
    parametrization_type="linear", min_val=20, max_val=100, unit=u.MW,
    param_points=1, name="P_aux"
)

P_lost_rad_all_DT_field = ParameterField(
    parametrization_type="linear", min_val=0, max_val=20, unit=u.MW,
    param_points=1, name="P_lost_rad"
)

print("Plasma parameter fields:")
print(f"V_plasma: {V_plasma_field}")
print(f"T_i_field: {T_i_field}")
print(f"n_tot_field: {n_tot_field}")
print(f"tau_p_T: {tau_p_T_field}")
print(f"tau_p_He3: {tau_p_He3_field}")
print(f"P_aux: {P_aux_field}")
print(f"P_aux_all_DT: {P_aux_all_DT_field}")
print(f"P_lost_rad: {P_lost_rad_field}")
print(f"P_lost_rad_all_DT: {P_lost_rad_all_DT_field}")

TBR_DT_field = ParameterField(
    parametrization_type="linear", min_val=1.05, max_val=1.15,
    param_points=2, name="TBR_DT"
)

TBR_DDn_field = ParameterField(
    parametrization_type="linear", min_val=0.5, max_val=0.9,
    param_points=3, name="TBR_DDn"
)

tau_ifc_field = ParameterField(
    parametrization_type="linear", min_val=1, max_val=12, unit=u.h,
    param_points=3, name="tau_ifc"
)

tau_ofc_field = ParameterField(
    parametrization_type="linear", min_val=1, max_val=24, unit=u.h,
    param_points=3, name="tau_ofc"
)

print("Breeding parameters:")
print(f"TBR_DT: {TBR_DT_field}")
print(f"TBR_DDn: {TBR_DDn_field}")
print(f"tau_ifc: {tau_ifc_field}")
print(f"tau_ofc: {tau_ofc_field}")

# Economic parameters
eta_th_field = ParameterField(
    parametrization_type="linear", min_val=0.3, max_val=0.4,
    param_points=2, name="eta_th"
)

plant_avail_field = ParameterField(
    parametrization_type="linear", min_val=0.5, max_val=0.9,
    param_points=5, name="plant_availability"
)

Cost_per_kWh_field = ParameterField(
    parametrization_type="normal", mean=0.25, std=0.15, unit=1/u.kWh,
    param_points=5, name="Cost_per_kWh"
)

print("Economic parameters:")
print(f"eta_th: {eta_th_field}")
print(f"plant_avail: {plant_avail_field}")
print(f"Cost_per_kWh: {Cost_per_kWh_field}")

Plasma parameter fields:
V_plasma: <ParameterField 'plasma_volume' type=normal, profile=none, shape=(1,), unit=meter ** 3>
[150.000] meter ** 3
T_i_field: <ParameterField 'T_i_field' type=linear, profile=none, shape=(3,), unit=kiloelectron_volt>
[14.000 17.000 20.000] kiloelectron_volt
n_tot_field: <ParameterField 'n_tot_field' type=linear, profile=none, shape=(3,), unit=1 / meter ** 3>
[130000000000000000000.000 170000000000000000000.000 210000000000000000000.000] 1 / meter ** 3
tau_p_T: <ParameterField 'tau_p_T' type=linear, profile=none, shape=(3,), unit=second>
[0.100 2.550 5.000] second
tau_p_He3: <ParameterField 'tau_p_He3' type=normal, profile=none, shape=(1,), unit=second>
[1.000] second
P_aux: <ParameterField 'P_aux' type=linear, profile=none, shape=(1,), unit=megawatt>
[60.000] megawatt
P_aux_all_DT: <ParameterField 'P_aux' type=linear, profile=none, shape=(1,), unit=megawatt>
[60.000] megawatt
P_lost_rad: <ParameterField 'P_lost_rad' type=linear, profile=none, shape=(3,), un

# Perform the parametric analysis

In [4]:
input_data = [
    V_plasma_field.data,
    T_i_field.data,
    n_tot_field.data,
    tau_p_T_field.data, 
    tau_p_He3_field.data,
    P_aux_field.data,
    P_lost_rad_field.data,
    P_aux_all_DT_field.data,
    P_lost_rad_all_DT_field.data,
        
    TBR_DT_field.data,
    TBR_DDn_field.data,
    tau_ifc_field.data,
    tau_ofc_field.data,
    
    eta_th_field.data,
    plant_avail_field.data,
    Cost_per_kWh_field.data,
]
# Create iterator based only on the number of parameter variations (first dimension)
param_ranges = [range(data.shape[0]) for data in input_data]

print(f"Total number of parameter combinations: {np.prod([len(r) for r in param_ranges])}")
# print estimated time to run (with 2 iterations/s)
print(f"Estimated time to perform the analysis: {np.prod([len(r) for r in param_ranges])/(1.5*3600)} hours (assuming 1.5it/s)")

Total number of parameter combinations: 218700
Estimated time to perform the analysis: 40.5 hours (assuming 1.5it/s)


# inventory ode system

In [5]:
def tritium_inventory_odes(t,y):
    N_ofc = y[0] # total number of tritium atoms in the outer fuel cycle
    N_ifc = y[1] # total number of tritium atoms in the inner fuel cycle
    N_st = y[2]  # total number of tritium atoms in the storage
    n_T = y[3]  # total tritium density in the plasma (float)
    
    n_T = n_T/u.m**3
    
    # Single spatial point - no integration needed
    n_D = n_tot - n_T
    Tdot_DDn = (TBR_DDn*0.5*n_D**2*sigmav_DD_n*V_plasma).to('1/s')
    Tdot_DDp = (0.5*n_D**2*sigmav_DD_p*V_plasma).to('1/s')
    Tdot_DT = (TBR_DT*n_D*n_T*sigmav_DT*V_plasma).to('1/s')
    Tdot_burn = (n_D*n_T*sigmav_DT*V_plasma).to('1/s')
    
    injection_rate = injection_rate_fun(N_ifc, N_st, tau_ifc, injection_rate_max=injection_rate_max)
   
    dN_ofc_dt = (Tdot_DT + Tdot_DDn - N_ofc / tau_ofc - N_ofc*lambda_T).to('1/s')
    dN_ifc_dt = (N_ofc / tau_ofc - N_ifc / tau_ifc  - lambda_T * N_ifc + n_T/tau_p_T*V_plasma).to('1/s')
    dN_stor_dt = (N_ifc / tau_ifc - lambda_T * N_st - injection_rate).to('1/s')
    dnT_dt = (injection_rate/V_plasma + Tdot_DDp/V_plasma - n_T/tau_p_T - Tdot_burn/V_plasma).to('1/s/m^3')

    total_T_produced = (Tdot_DDn + Tdot_DDp + Tdot_DT).to('1/s')
    total_T_burnt = (n_D*n_T*sigmav_DT*V_plasma).to('1/s')

    net_T_produced = total_T_produced - total_T_burnt

    return [float(dN_ofc_dt.to('1/s').magnitude), float(dN_ifc_dt.to('1/s').magnitude), float(dN_stor_dt.to('1/s').magnitude), float(dnT_dt.to('1/s/m^3').magnitude)]


In [ ]:
results = []
    
def DT_reached(t, y):
    n_T = y[3]/u.m**3
    # Stop when total tritium reaches 50% of total inventory
    return (n_T.to('1/m^3') - 0.5 * n_tot.to('1/m^3')).magnitude

def NEGATIVE(t, y):
    # stop when y[0] or y[1] or y[2] or y[3] is negative
    return min(y[0]+1e-10, y[1]+1e-10, y[2]+1e-10, y[3]+1e-10)


# Time span
t_span = (0, total_time.to('s').magnitude)
t_eval = np.linspace(*t_span, 1000)
    
for param_combo in tqdm(itertools.product(*param_ranges), 
                       total=np.prod([data.shape[0] for data in input_data]), 
                       desc="Parametric analysis"):
    
    # Extract data for each parameter
    extracted_data = [input_data[i][param_idx] for i, param_idx in enumerate(param_combo)]
        
    # Unpack the extracted data
    (V_plasma, T_i, n_tot,
     tau_p_T, tau_p_He3, 
     P_aux, P_lost_rad, P_aux_all_DT, P_lost_rad_all_DT,
     
     TBR_DT, TBR_DDn, tau_ifc, tau_ofc,
     
     eta_th, plant_avail, Cost_per_kWh,
     ) = extracted_data


    # Get cross-sections
    sigmav_DD = sigmav_DD_BoschHale(T_i)[0].to('m^3/s')  # [m^3/s]
    sigmav_DD_p = sigmav_DD_BoschHale(T_i)[1].to('m^3/s')  # [m^3/s]
    sigmav_DD_n = sigmav_DD_BoschHale(T_i)[2].to('m^3/s')  # [m^3/s]
    sigmav_DT = sigmav_DT_BoschHale(T_i).to('m^3/s')    # [m^3/s]
    sigmav_DHe3 = sigmav_DHe3_BoschHale(T_i).to('m^3/s')   # [m^3/s]
    
    y0 = np.zeros(3 + n_tot.size)  # Initial conditions: [N_ofc, N_ifc, N_st, n_T]

    # stop whe  DT reached
    DT_reached.terminal = True
    NEGATIVE.terminal = True
    
    N_st_min = 0.001/tritium_mass.to('kg').magnitude
    # maximum injection rate is the injection rate necessary to maintain a 50D50T plasma
    injection_rate_max = (n_tot/2/tau_p_T*V_plasma + 0.25*n_tot**2*sigmav_DT*V_plasma - 0.25/2*n_tot**2*sigmav_DD_p*V_plasma).to('1/s')
    
    # solve system of ODEs
    sol = solve_ivp(
        fun = tritium_inventory_odes, 
        t_span = t_span, 
        t_eval = t_eval,
        y0 = y0, 
        method = 'BDF', 
        dense_output=False,
        events = [DT_reached, NEGATIVE], )
    
    # identify t_startup
    if sol.t_events[1].size > 0:
        print(f"Negative event occurred at t={sol.t_events[1]}")
        break
    elif sol.t_events[0].size > 0:
        t_startup = sol.t_events[0][0]*u.s
    else:
        t_startup = np.inf*u.s
        
    n_T = sol.y[3] * u.m**(-3)  # shape (n_spatial, n_time)
    n_D = n_tot - n_T
    
    # evolution of fusion power
    P_DDn = n_D*n_D*sigmav_DD_n/2*V_plasma*E_DDn
    P_DDp = n_D*n_D*sigmav_DD_p/2*V_plasma*E_DDp
    P_DT = n_D*n_T*sigmav_DT*V_plasma*E_DT
    # equivalent energy if always DT
    P_DT_full = n_tot/2*n_tot/2*sigmav_DT*V_plasma*E_DT
    
    t_end = t_startup.to('s').magnitude if np.isfinite(t_startup.to('s').magnitude) else total_time.to('s').magnitude
    mask = sol.t <= t_end

    net_energy_DD = eta_th*np.trapz((P_DDn[mask]+P_DDp[mask]+P_DT[mask]-P_lost_rad*np.ones(sol.t[mask].shape))-P_aux*np.ones(sol.t[mask].shape), sol.t[mask]*u.s)
    net_energy_DT_full = eta_th*np.trapz((P_DT_full-P_lost_rad_all_DT)*np.ones(sol.t[mask].shape)-P_aux_all_DT*np.ones(sol.t[mask].shape), sol.t[mask]*u.s)

    P_e_net_DD, Q_DD = calculate_P_e_net((P_DDn+P_DDp+P_DT),P_aux=P_aux, P_rad = P_lost_rad,  plant_avail=plant_avail, eta_th=eta_th)
    P_e_net_DT_full, Q_DT_full = calculate_P_e_net(P_DT_full, P_aux=P_aux_all_DT, P_rad=P_lost_rad_all_DT, plant_avail=plant_avail, eta_th=eta_th)
    E_e_net_DD = np.trapz(P_e_net_DD[mask], sol.t[mask]*u.s)
    E_e_net_DT_full = P_e_net_DT_full*sol.t[mask]*u.s
    
    # energy lost
    # Integrate over time (up to t_startup if not infinite, else total_time)
    E_lost = E_e_net_DT_full - E_e_net_DD
    Dollar_Lost = E_lost * Cost_per_kWh
    
    n_T = sol.y[3] * u.m**(-3)
    I_ifc = sol.y[0] * tritium_mass.to('kg')
    I_ofc = sol.y[1] * tritium_mass.to('kg')
    I_stor = sol.y[2] * tritium_mass.to('kg')
    
    row = [
        # INPUTS
        V_plasma.to('m^3'),                     # 0
        tau_p_T.to('s'),                        # 1
        tau_p_He3.to('s'),                      # 2
        P_aux.to('MW'),                         # 3
        P_aux_all_DT.to('MW'),                  # 4
        P_lost_rad.to('MW'),                    # 5
        P_lost_rad_all_DT.to('MW'),             # 6
        T_i.to('keV'),                          # 7
        n_tot.to('m^-3'),                       # 8
        #injection_rate_max,                    # -
        TBR_DT,                                 # 9
        TBR_DDn,                                # 10
        tau_ifc,                                # 11
        tau_ofc,                                # 12
        eta_th,                                 # 13
        plant_avail,                            # 14
        Cost_per_kWh.to('1/kWh'),               # 15
        # OUTPUTS
        sigmav_DT.to('m^3/s'),                  # 16
        sigmav_DD_n.to('m^3/s'),                # 17
        sigmav_DD_p.to('m^3/s'),                # 18
        sigmav_DHe3.to('m^3/s'),                # 19
        P_DT.to('MW'),                          # 20
        P_DDn.to('MW'),                         # 21
        P_DDp.to('MW'),                         # 22
        P_DT_full.to('MW'),                     # 23
        P_e_net_DD.to('MW'),                    # 24
        Q_DD,                                   # 25
        P_e_net_DT_full.to('MW'),               # 26
        Q_DT_full,                              # 27
        E_e_net_DD.to('MJ'),                    # 28
        E_e_net_DT_full.to('MJ'),               # 29
        t_startup.to('hour'),                   # 30
        E_lost.to('MJ'),                        # 31
        Dollar_Lost.to(''),                     # 32
        n_T,                                    # 33
        I_ifc,                                  # 34
        I_ofc,                                  # 35
        I_stor                                  # 36
    ]
    results.append(row)

Parametric analysis:   0%|          | 54/218700 [01:52<99:10:21,  1.63s/it] 

# save to csv

In [ ]:
# save results to a DataFrame
columns = [
    # INPUTS
    "V_plasma (m^3)",                     # 0
    "tau_p_T (s)",                        # 1
    "tau_p_He3 (s)",                      # 2
    "P_aux (MW)",                         # 3
    "P_aux_all_DT (MW)",                  # 4
    "P_lost_rad (MW)",                    # 5
    "P_lost_rad_all_DT (MW)",             # 6
    "T_i (keV)",                          # 7   
    "n_tot (m^-3)",                       # 8
    #"injection_rate_max",                 # -
    "TBR_DT",                             # 9
    "TBR_DDn",                            # 10
    "tau_ifc (h)",                        # 11
    "tau_ofc (h)",                        # 12  
    "eta_th",                             # 13
    "plant_avail",                        # 14
    "Cost_per_kWh (1/kWh)",               # 15
    # OUTPUTS
    "P_DT (MW)",                          # 16
    "P_DDn (MW)",                         # 17
    "P_DDp (MW)",                         # 18  
    "P_DT_full (MW)",                     # 19
    "t_startup (h)",                      # 20
    "E_lost (MJ)",                        # 21
    "Dollar_Lost ($)",                    # 22
    "n_T (m^-3)",                         # 23
    "I_ifc (kg)",                         # 24
    "I_ofc (kg)",                         # 25
    "I_stor (kg)"                         # 26
]

df_results = pd.DataFrame(results, columns=columns)
df_results.to_csv("parametric_study_results_taup=0.1.csv", index=False)